# Market Basket Analysis
This notebook performs Market Basket Analysis on the vending machine dataset.
Baskets are formed by grouping transactions that occur within 1 minute of each other.

In [4]:
import warnings
# Filter specific DeprecationWarnings from jupyter_client
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client')

from google.colab import drive
from google.colab import files
import pandas as pd
import numpy as np
import io
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Mount Google Drive (optional, if you want to save/load directly from Drive)
drive.mount('/content/drive')

# Prompt user to upload the dataset
print("Please upload your CSV dataset:")
uploaded = files.upload()

# Get the exact filename of the uploaded file dynamically
filename = next(iter(uploaded))

# Load the dataset (now without skipping the first row, assuming it's the header)
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Convert 'Transaction Date' to datetime objects (handles formats like '2025-11-27 12:46:32')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

# Sort by Transaction Date to ensure chronological order
df = df.sort_values('Transaction Date').reset_index(drop=True)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please upload your CSV dataset:


Saving MACHINE_B_OCTOBER_FINAL.csv to MACHINE_B_OCTOBER_FINAL (3).csv


,Group Name,Machine,Transaction Date,Transaction No,Slot No,Product Code,Menu,Service,Status,Delivery,Category,Price
0,Master,Machine B,2025-10-01 09:22:50,2500724,6,UMMIB2,Berry Sweet Tart Pot,Cash,Successful,Delivery,Vegetarian,Affordable
1,Master,Machine B,2025-10-01 09:42:58,2501597,17,UMMIF1,Banana Berry Acai Bowl,Cash,Successful,Delivery,Vegetarian,Affordable
2,Master,Machine B,2025-10-01 09:44:08,2501719,1,UMMIJ4,Pomegranate Juice,Cash,Successful,Delivery,Beverage,Affordable
3,Master,Machine B,2025-10-01 09:53:56,2504591,17,UMMIF1,Banana Berry Acai Bowl,Cash,Successful,Delivery,Vegetarian,Affordable
4,Master,Machine B,2025-10-01 10:06:05,2505086,6,UMMIB2,Berry Sweet Tart Pot,Cash,Successful,Delivery,Vegetarian,Affordable


## Create Baskets based on Time Window
We group consecutive transactions into the same basket if they occur within 1 minute (60 seconds) of each other.

In [5]:
# Calculate time difference between consecutive transactions
df['Time Diff'] = df['Transaction Date'].diff().dt.total_seconds()

# A new basket starts if the time difference is greater than 120 seconds (or is NaN for the first row)
df['New Basket'] = (df['Time Diff'] > 120) | df['Time Diff'].isna()

# Create a Basket ID by taking the cumulative sum of the 'New Basket' boolean column
df['Basket ID'] = df['New Basket'].cumsum()

print(f"Total rows: {len(df)}")
print(f"Total unique baskets: {df['Basket ID'].nunique()}")

# Extract the baskets
baskets = df.groupby('Basket ID')['Menu'].apply(list).reset_index()
baskets_list = baskets['Menu'].tolist()

baskets.head()

Total rows: 397
Total unique baskets: 308


,Basket ID,Menu
0,1,[Berry Sweet Tart Pot]
1,2,"[Banana Berry Acai Bowl , Pomegranate Juice]"
2,3,[Banana Berry Acai Bowl ]
3,4,"[Berry Sweet Tart Pot, Mixed Vegetables with P..."
4,5,[Tomato Mozzarella Pesto Sandwich ]


## One-Hot Encoding
Convert the list of baskets into a one-hot encoded format required by the `mlxtend` library.

In [6]:
te = TransactionEncoder()
te_ary = te.fit(baskets_list).transform(baskets_list)

# Create a DataFrame from the encoded array
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

basket_df.head()

,Apple Green Juice Fresh,Avo-licious Pot,BBQ Prawns with Fusilli Pasta,Banana Berry Acai Bowl,Berry Sweet Tart Pot,Carrot Juice Fresh,Chicken & Yuzu Poke Bowl,Chicken Biryani,Chicken Brown & Wild Rice Salad,Chicken Caesar Wrap,...,Spicy Chicken with Pearl CousCous,Steamed Rice with Butter Chicken,Strawberry Smoothie,SuperFood with Cheese Salad,Sweet And Sour Prawn Teriyaki Noodles,Tomato Mozzarella Pesto Sandwich,Tropical Açaí Bowl,Vegan Burrito Bowl,Vegetable Crudités With Labneh Herb Dip,Watermelon Juice Fresh
0,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


## Frequent Itemsets using Apriori
Find frequent itemsets with a specified minimum support.

In [7]:
# Adjust min_support based on your dataset size and needs
frequent_itemsets = apriori(basket_df, min_support=0.005, use_colnames=True)

# Sort by support
frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)

frequent_itemsets.head(10)

,support,itemsets
6,0.081169,(Chicken & Yuzu Poke Bowl)
32,0.077922,(Pomegranate Juice)
9,0.055195,(Chicken Caesar Wrap )
34,0.055195,(Prawn Mango Salad)
44,0.051948,(Watermelon Juice Fresh)
17,0.048701,(Dynamite Chicken Sandwich on Brown Bread)
13,0.048701,(Chicken Tikka Club with Brown Bread)
40,0.045455,(Tomato Mozzarella Pesto Sandwich )
19,0.045455,(Fajita Chicken Wrap)
30,0.042208,(Pasta & Herb Chicken Salad)


## Association Rules
Generate association rules to find relationships between products.

In [8]:
# Generate association rules.
# We set a minimum threshold for lift > 1.0 (indicating that the items are positively correlated).
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Sort the generated rules by confidence and lift to find the strongest rules
rules = rules.sort_values(['confidence', 'lift'], ascending=[False, False])

# Display the top 10 rules
display(rules.head(10))

# GENERATE AND DOWNLOAD CSV
output_csv = 'MBA_Combinations_Output.csv'

# Convert frozensets to standard strings so they look clean in the CSV
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Save to CSV
rules.to_csv(output_csv, index=False)
print(f"\nSuccessfully generated {output_csv}!")

# Automatically trigger the download in Google Colab
files.download(output_csv)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
6,(Apple Green Juice Fresh),(Falafel Wrap),0.006494,0.019481,0.006494,1.000000,51.333333,1.0,0.006367,inf,0.986928,0.333333,1.000000,0.666667
30,"(Watermelon Juice Fresh, Pomegranate Juice)",(Chicken Biryani),0.006494,0.035714,0.006494,1.000000,28.000000,1.0,0.006262,inf,0.970588,0.181818,1.000000,0.590909
32,"(Pomegranate Juice, Chicken Biryani)",(Watermelon Juice Fresh),0.006494,0.051948,0.006494,1.000000,19.250000,1.0,0.006156,inf,0.954248,0.125000,1.000000,0.562500
31,"(Watermelon Juice Fresh, Chicken Biryani)",(Pomegranate Juice),0.006494,0.077922,0.006494,1.000000,12.833333,1.0,0.005988,inf,0.928105,0.083333,1.000000,0.541667
22,(Lemonade Juice Fresh),(Chicken Brown & Wild Rice Salad ),0.016234,0.038961,0.006494,0.400000,10.266667,1.0,0.005861,1.601732,0.917492,0.133333,0.375676,0.283333
5,(Carrot Juice Fresh),(Chicken Tikka Club with Brown Bread),0.016234,0.048701,0.006494,0.400000,8.213333,1.0,0.005703,1.585498,0.892739,0.111111,0.369283,0.266667
7,(Falafel Wrap),(Apple Green Juice Fresh),0.019481,0.006494,0.006494,0.333333,51.333333,1.0,0.006367,1.490260,1.000000,0.333333,0.328976,0.666667
29,(Strawberry Smoothie),(Watermelon Juice Fresh),0.019481,0.051948,0.006494,0.333333,6.416667,1.0,0.005482,1.422078,0.860927,0.100000,0.296804,0.229167
0,(Fajita Chicken Wrap),(Prawn Mango Salad),0.045455,0.055195,0.012987,0.285714,5.176471,1.0,0.010478,1.322727,0.845238,0.148148,0.243986,0.260504
15,(Chicken Tandoori with Coriander Rice),(Chicken & Yuzu Poke Bowl),0.025974,0.081169,0.006494,0.250000,3.080000,1.0,0.004385,1.225108,0.693333,0.064516,0.183746,0.165000



Successfully generated MBA_Combinations_Output.csv!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>